# Denoising Autoencoder & Latent Space Analysis

This notebook demonstrates:
- Unsupervised feature learning with autoencoders
- Denoising with Gaussian noise
- Dimensionality reduction to latent space
- Supervised learning on latent features
- Latent space visualization

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(''))))

# Change to script directory
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(''))))

# Import training script
from training.train_autoencoder import main as train_ae_main

print("Autoencoder training script imported successfully!")

## Step 1: Autoencoder Concepts

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║          AUTOENCODER ARCHITECTURE                                 ║
╚════════════════════════════════════════════════════════════════════╝

An autoencoder learns to reconstruct its input through a bottleneck.

ARCHITECTURE:

Input (8 features)
    ↓
ENCODER: (Compression)
    ├─ Dense(64) + ReLU
    ├─ Dense(32) + ReLU
    └─ Dense(16) + ReLU  ← LATENT SPACE (bottleneck)
                           (compressed representation)
    ↓
DECODER: (Reconstruction)
    ├─ Dense(32) + ReLU
    ├─ Dense(64) + ReLU
    └─ Output(8)  ← Reconstructed input

═══════════════════════════════════════════════════════════════════════

KEY COMPONENTS:

1. ENCODER (Compression Phase):
   - Reduces 8 features → 16 latent dimensions
   - Compression ratio: 8/16 = 0.5x
   - Learns essential features that preserve information
   - Non-linear transformations via ReLU

2. LATENT SPACE (Bottleneck):
   - 16-dimensional compressed representation
   - Forced to capture most important information
   - Dimensionality reduction: 8 → 16 dimensions
   - Acts as feature compression

3. DECODER (Reconstruction Phase):
   - Reconstructs 16 latent dims → 8 features
   - Must recover as much information as possible
   - Linear output layer (no activation)

4. LOSS FUNCTION:
   - Reconstruction MSE: mean((input - reconstructed)²)
   - Measures how well latent space captures information
   - Lower loss = better compression and reconstruction

═══════════════════════════════════════════════════════════════════════

DENOISING MECHANISM:

TRAINING:
1. Add Gaussian noise to input: x_noisy = x + N(0, 0.1)
2. Feed noisy input to autoencoder
3. Autoencoder learns to reconstruct CLEAN input

BENEFITS:
- Prevents trivial solution (copying input unchanged)
- Improves robustness to noise
- Forces learning of essential features
- Better generalization
- Regularization effect similar to dropout

TESTING:
- Use clean input (no noise added)
- Encoder extracts latent features
- Latent features are used for prediction
""")

## Step 2: Train Autoencoder

In [ ]:
# Execute the training script
autoencoder, encoder, mlp_model, ae_time, mlp_time = train_ae_main()

print(f"\nAutoencoder training completed!")
print(f"Autoencoder training time: {ae_time:.2f} seconds")
print(f"MLP on latent features time: {mlp_time:.2f} seconds")

## Step 3: Understanding Latent Space

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║         LATENT SPACE & FEATURE EXTRACTION                         ║
╚════════════════════════════════════════════════════════════════════╝

WHAT IS LATENT SPACE?

- Compressed representation of input data
- Lower-dimensional space (8 features → 16 latent dims)
- Contains essential information about the data
- Learned by the encoder in unsupervised manner

PROPERTIES OF LATENT SPACE:

1. DIMENSIONALITY REDUCTION:
   - Original: 8 features
   - Latent: 16 dimensions
   - Compression captures data structure efficiently

2. INFORMATION PRESERVATION:
   - Latent features enable reconstruction of input
   - High reconstruction accuracy → good feature preservation
   - Can also be used for downstream prediction

3. LEARNED WITHOUT LABELS:
   - Unsupervised learning (no y values needed for AE)
   - Learns from data structure alone
   - Can be used with ANY labeled data for prediction

4. INTERPRETABILITY:
   - Each latent dimension may capture different aspect
   - E.g., one dim might capture "engine power"
   - Another might capture "vehicle weight"
   - Unsupervised but meaningful representations

═══════════════════════════════════════════════════════════════════════

TWO-PHASE TRAINING:

PHASE 1: AUTOENCODER TRAINING (Unsupervised)
  Input: X_train (no targets needed)
  Task: Minimize reconstruction loss
  Output: Trained encoder and decoder
  
  Loss = mean((X_train - reconstructed_X)²)

PHASE 2: MLP ON LATENT FEATURES (Supervised)
  Input: Latent features from encoder
  Output: MPG predictions
  Task: Minimize prediction loss
  Loss = mean((y_train - predictions)²)
  
  This proves latent features are useful for prediction!

WHY DOES THIS WORK?

- Encoder learns data structure without seeing targets
- Latent space captures input variations
- These variations correlate with MPG
- MLP leverages learned representations for prediction
- Often works better than training MLP on original data
  (acts as regularization, prevents overfitting)
""")

## Step 4: Visualizations

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Display all generated plots
image_files = [
    'images/autoencoder/Autoencoder_reconstruction_loss.png',
    'images/autoencoder/Autoencoder_latent_space_pca.png',
    'images/autoencoder/MLP-on-Latent_training_history.png'
]

for img_path in image_files:
    if os.path.exists(img_path):
        img = Image.open(img_path)
        plt.figure(figsize=(12, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(img_path.split('/')[-1])
        plt.tight_layout()
        plt.show()
    else:
        print(f"Image not found: {img_path}")

## Step 5: Advanced Autoencoder Concepts

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║     AUTOENCODER VARIANTS & INTERPRETATIONS                        ║
╚════════════════════════════════════════════════════════════════════╝

TYPES OF AUTOENCODERS:

1. STANDARD AUTOENCODER:
   - Reconstructs input from latent space
   - Loss: reconstruction MSE
   - Use: compression, feature extraction

2. DENOISING AUTOENCODER (used here):
   - Removes noise from inputs
   - Enhanced robustness
   - Better generalization

3. VARIATIONAL AUTOENCODER (VAE):
   - Probabilistic model
   - Latent space is distribution, not deterministic
   - Can generate new data by sampling latent space

4. SPARSE AUTOENCODER:
   - Enforces sparsity in latent layer
   - Promotes interpretability
   - Only few latent units active

═══════════════════════════════════════════════════════════════════════

WHY LATENT SPACE HELPS WITH PREDICTION:

HYPOTHESIS 1: Dimensionality Reduction
- Lower dimensions reduce noise
- Focus on principal variations
- Avoid overfitting from high-dimensional space

HYPOTHESIS 2: Feature Learning
- Encoder learns non-linear combinations
- More useful representations than original features
- Captures interactions between variables

HYPOTHESIS 3: Regularization
- Forced compression acts as regularization
- Bottleneck prevents overfitting
- Similar effect to dropout or L2 regularization

═══════════════════════════════════════════════════════════════════════

PCA VISUALIZATION OF LATENT SPACE:

- Latent space: 16 dimensions (too many to visualize)
- PCA reduces to 2D for visualization
- Dots represent encoded samples
- Color represents sample index
- Clustering patterns indicate learned structure
- Spread indicates diversity captured by encoder
""")

## Step 6: Key Takeaways

In [ ]:
print(f"""
╔════════════════════════════════════════════════════════════════════╗
║        AUTOENCODER TRAINING SUMMARY                              ║
╚════════════════════════════════════════════════════════════════════╝

Model 1: Denoising Autoencoder
  Architecture: 8 → 64 → 32 → 16 → 32 → 64 → 8
  Latent Dimension: 16
  Training Time: {ae_time:.2f} seconds
  Task: Reconstruct clean input from noisy input

Model 2: MLP on Latent Features
  Architecture: 16 → 32 → 16 → 1
  Input: Latent features (16 dims)
  Output: MPG prediction
  Training Time: {mlp_time:.2f} seconds
  Task: Predict MPG from latent features

Key Concepts Demonstrated:
1. Unsupervised feature learning with autoencoders
2. Denoising with Gaussian noise injection
3. Dimensionality reduction (8 → 16 latent)
4. Reconstruction loss optimization
5. Encoder extraction for downstream tasks
6. Two-phase training: unsupervised + supervised
7. Latent space visualization with PCA
8. Regularization through bottleneck compression

Applications:
- Anomaly detection (high reconstruction error = anomaly)
- Data compression and storage
- Feature extraction for other tasks
- Noise removal preprocessing
- Visualization of high-dimensional data
""")